# Proyecto de demanda 

## Objetivo Específico 3:

### Determinar la demanda laboral a nivel nacional, regional y local mediante técnicas de analítica de datos con  el propósito de orientar la toma de decisiones sobre la oferta académica de la UNL

## Resultados Esperados

### Reporte semestral sobre la demanda laboral...

### Metodología: web scrapping, APIs y text mining




Multitrabajos piloto

- Crear entorno virtual local
- Activar entorno virtual


Correr en la terminal:

python3.11 -m venv cise_scraper
cise_scraper\Scripts\activate

- Instalar dependencias

In [ ]:
import subprocess, sys

paquetes = [
    "undetected-chromedriver",
    "selenium",
    "beautifulsoup4",
    "requests",
    "pandas",
    "cloudscraper",
    "setuptools",
]

for pkg in paquetes:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])
    print(f"✅ {pkg}")

✅ undetected-chromedriver
✅ selenium
✅ beautifulsoup4
✅ requests
✅ pandas
✅ cloudscraper
✅ setuptools


In [20]:
# ============================================================
# CELDA 2: Configuración — ajustar DB_PATH
# ============================================================
import os

# ── ÚNICO PARÁMETRO QUE DEBES AJUSTAR ────────────────────────
DB_PATH = r"C:\Users\alexis\Downloads\CISE_2026\vacantes_laborales.db" #aqui colocar ruta local

# Crear carpeta si no existe
os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)
print(f"✅ BD apuntando a: {DB_PATH}")
print(f"✅ Carpeta existe : {os.path.exists(os.path.dirname(DB_PATH))}")

✅ BD apuntando a: C:\Users\alexis\Downloads\CISE_2026\vacantes_laborales.db
✅ Carpeta existe : True


In [41]:
# ============================================================
# CELDA 3: Schema BD — copiar sin cambios desde Colab
# ============================================================
import sqlite3, hashlib, re
from datetime import datetime

def conectar():
    """Retorna conexión a la BD local."""
    return sqlite3.connect(DB_PATH)

def crear_schema():
    conn = conectar()
    cur  = conn.cursor()

    cur.executescript("""
    CREATE TABLE IF NOT EXISTS portales (
        id            INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre        TEXT    UNIQUE NOT NULL,
        url_base      TEXT,
        activo        INTEGER DEFAULT 1,
        fecha_alta    TEXT    DEFAULT (datetime('now'))
    );

    CREATE TABLE IF NOT EXISTS ejecuciones (
        id              INTEGER PRIMARY KEY AUTOINCREMENT,
        portal_id       INTEGER REFERENCES portales(id),
        fecha_inicio    TEXT,
        fecha_fin       TEXT,
        total_vacantes  INTEGER DEFAULT 0,
        estado          TEXT    DEFAULT 'en_curso'
    );

    CREATE TABLE IF NOT EXISTS vacantes (
        id                  INTEGER PRIMARY KEY AUTOINCREMENT,
        hash_id             TEXT    UNIQUE NOT NULL,
        portal_id           INTEGER REFERENCES portales(id),
        ejecucion_id        INTEGER REFERENCES ejecuciones(id),

        -- Campos raw
        cargo_raw           TEXT,
        empresa_raw         TEXT,
        ubicacion_raw       TEXT,
        salario_raw         TEXT,
        modalidad_raw       TEXT,
        fecha_publicacion   TEXT,
        url_detalle         TEXT,
        descripcion_raw     TEXT,

        -- Campos normalizados
        cargo_norm          TEXT,
        empresa_norm        TEXT,

        -- Codificación estándar
        codigo_dpa          TEXT,
        codigo_ciiu         TEXT,
        codigo_ciuo         TEXT,

        -- Condiciones laborales
        salario_min         REAL,
        salario_max         REAL,
        salario_a_convenir  INTEGER DEFAULT 0,
        experiencia_anos    INTEGER,

        -- Flags de procesamiento
        ciiu_procesado      INTEGER DEFAULT 0,
        ciuo_procesado      INTEGER DEFAULT 0,
        xgb_imputado        INTEGER DEFAULT 0,
        subarea_raw         TEXT,
        texto_raw           TEXT,

        fecha_extraccion    TEXT
    );
    """)

    # Insertar portales
    portales = [
        ("multitrabajos",   "https://www.multitrabajos.com/empleos"),
        ("computrabajo",    "https://ec.computrabajo.com/ofertas-de-trabajo"),
        ("encuentraempleo", "https://www.encuentraempleo.com.ec"),
        ("socioempleo",     "https://socioempleo.gob.ec"),
        ("bebee",           "https://www.bebee.com/jobs/ec"),
    ]
    cur.executemany(
        "INSERT OR IGNORE INTO portales (nombre, url_base) VALUES (?,?)",
        portales
    )
    conn.commit()
    conn.close()
    print("✅ Schema creado y portales insertados")

crear_schema()

✅ Schema creado y portales insertados


In [54]:
# ============================================================
# CELDA 4: Driver — versión local sin flags de Colab
# ============================================================
import time, random
import undetected_chromedriver as uc


# headless=False obligatorio para Multitrabajos/Computrabajo
# Cloudflare detecta flags de Chromium headless → bloqueo 403

def crear_driver(headless=True):
    """
    En local, uc.Chrome() detecta Chrome automáticamente.
    
    headless=False → puedes VER el browser (útil para depurar)
    headless=True  → producción normal
    """
    opts = uc.ChromeOptions()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--window-size=1920,1080")
    opts.add_argument("--lang=es-EC")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_argument(
        "--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/147.0.0.0 Safari/537.36"
    )
    driver = uc.Chrome(options=opts, version_main=147) #ajustar version de google chrome aqui
    driver.execute_script(
        "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    )
    return driver

def espera_humana(mn=2.0, mx=5.0):
    time.sleep(random.uniform(mn, mx))

print("✅ Driver local configurado")
print("💡 Tip: crear_driver(headless=False) para ver el browser en acción")

✅ Driver local configurado
💡 Tip: crear_driver(headless=False) para ver el browser en acción


In [56]:
# ============================================================
# CELDA PRUEBA
# ============================================================
driver = crear_driver(headless=True)
try:
    driver.get("https://www.multitrabajos.com/empleos.html")
    time.sleep(6)
    print(f"Título : {driver.title!r}")
    print(f"¿CF?   : {'cf-wrapper' in driver.page_source}")
    print(f"Bytes  : {len(driver.page_source):,}")
finally:
    driver.quit()

Título : 'Empleos en Ecuador | Ofertas de Trabajo - Página 1 | Multitrabajos'
¿CF?   : False
Bytes  : 277,290


In [34]:
# ============================================================
# CELDA 5: Identificar selectores reales de Multitrabajos
# Con 279KB de HTML las vacantes YA están en el DOM
# ============================================================
from bs4 import BeautifulSoup
import time

driver = crear_driver(headless=False)  # False → se te abre una ventana de navegador

try:
    driver.get("https://www.multitrabajos.com/empleos.html")
    time.sleep(6)
    
    html  = driver.page_source
    soup  = BeautifulSoup(html, "html.parser")

    print(f"📄 Bytes recibidos: {len(html):,}")
    print(f"📄 Título: {driver.title}\n")

    # ── Buscar contenedores de vacantes ──────────────────────
    candidatos = [
        "article",
        "div[class*='box_offer']",
        "div[class*='offer']",
        "div[class*='job']",
        "li[class*='offer']",
        "li[class*='job']",
        "div[class*='card']",
        "div[class*='item']",
        "div[class*='result']",
        "section[class*='offer']",
        "[data-id]",
        "[data-jobid]",
        "[data-offer]",
    ]

    print("=" * 55)
    print("BÚSQUEDA DE CONTENEDOR DE VACANTES")
    print("=" * 55)
    for sel in candidatos:
        elementos = soup.select(sel)
        if elementos:
            clase = elementos[0].get("class", [])
            print(f"\n✅ '{sel}' → {len(elementos)} elementos")
            print(f"   Clase real : {' '.join(clase)}")
            print(f"   HTML snippet:")
            print(elementos[0].prettify()[:600])
            print("-" * 40)
        else:
            print(f"❌ '{sel}'")

    # ── Buscar texto de empleo en cualquier tag ───────────────
    print("\n" + "=" * 55)
    print("BÚSQUEDA POR TEXTO (cualquier tag con 'empleo')")
    print("=" * 55)
    
    # Todos los tags que contengan palabras clave laborales
    for tag in soup.find_all(True):
        texto = tag.get_text(strip=True)
        clases = " ".join(tag.get("class", []))
        if (len(texto) > 10 and len(texto) < 120 and
            any(k in texto.lower() for k in 
                ["ingenier", "contador", "vendedor", "asistente",
                 "analista", "gerente", "técnico", "supervisor"])):
            print(f"  <{tag.name} class='{clases}'> → {texto[:80]!r}")

finally:
    driver.quit()

📄 Bytes recibidos: 271,424
📄 Título: Trabajos en Ecuador - Empleos Multitrabajos 2020

BÚSQUEDA DE CONTENEDOR DE VACANTES
❌ 'article'
❌ 'div[class*='box_offer']'
❌ 'div[class*='offer']'
❌ 'div[class*='job']'
❌ 'li[class*='offer']'
❌ 'li[class*='job']'
❌ 'div[class*='card']'
❌ 'div[class*='item']'
❌ 'div[class*='result']'
❌ 'section[class*='offer']'
❌ '[data-id]'
❌ '[data-jobid]'
❌ '[data-offer]'

BÚSQUEDA POR TEXTO (cualquier tag con 'empleo')
  <div class='sc-ibnDSj eaWZbk'> → 'Contador General con experiencia en CostosConfidencial'
  <div class='sc-jtEaiv hlnIcg'> → 'Contador General con experiencia en CostosConfidencial'
  <span class='sc-OqFzE bLbfuP'> → 'Contador General con experiencia en Costos'
  <h2 class='sc-iaFXAv jUsAjO'> → 'Contador General con experiencia en Costos'
  <div class='sc-ibnDSj bOYzEK'> → 'Analista de ambienteNOVACERO S.A.4.6'
  <div class='sc-jtEaiv hPWmYU'> → 'Analista de ambienteNOVACERO S.A.4.6'
  <span class='sc-OqFzE bLbfuP'> → 'Analista de ambiente'
  <

In [ ]:
# ============================================================
# CELDA 6: PASO 1 — Extraer URLs y campos básicos del listado
# ============================================================
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from bs4 import BeautifulSoup
import time, random, sqlite3
from datetime import datetime
import re
URL_LISTADO = "https://www.multitrabajos.com/empleos.html"
BASE_URL    = "https://www.multitrabajos.com"




def scrape_listado_mt(driver, url: str, n_scrolls: int = 8) -> list[dict]:
    """
    Multitrabajos usa scroll infinito — no hay parámetro ?p=N.
    Hacemos N scrolls al fondo para forzar la carga de más tarjetas.
    n_scrolls=8 carga ~160 vacantes aprox (20 por scroll).
    """
    driver.get(url)

    try:
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "a[href*='empleo']"))
        )
    except TimeoutException:
        print("  ⚠️  Timeout inicial. Usando espera fija 8s...")
        time.sleep(8)

    urls_vistas = set()
    items       = []
    

    for scroll_n in range(n_scrolls):
        # Scroll al fondo de la página
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        espera_humana(2, 3)   # esperar que cargue el nuevo bloque

        soup     = BeautifulSoup(driver.page_source, "html.parser")
        tarjetas = soup.select("a[href*='/empleos/'][href$='.html']")

        # Fallback si el selector principal falla
        if not tarjetas:
            tarjetas = [a for a in soup.select("a[href*='empleo']")
                        if a.find("h2") or a.find("h3")]

        nuevas = 0
        for t in tarjetas:
            href = t.get("href", "")
            url_detalle = (BASE_URL + href
                           if href and not href.startswith("http") else href)

            if url_detalle in urls_vistas:
                continue   # ya la teníamos → skip
            urls_vistas.add(url_detalle)
            nuevas += 1

            def texto_icono(soup_t, icon_name: str) -> str:
                i = soup_t.find("i", attrs={"name": icon_name})
                if i:
                    li = i.find_parent("li") or i.find_parent("div")
                    if li:
                        el = li.find(["p", "h3", "span"])
                        if el:
                            return el.get_text(strip=True)
                return ""

            ubicacion = texto_icono(t, "icon-light-location-pin")
            modalidad = texto_icono(t, "icon-light-office")

            cargo = t.find("h2")
            empresa = ""
            _excluir = ("hace", "ayer", "hoy", "hora", "minuto",
                        "publicado", "nuevo", "destacado")
            for h in t.find_all("h3"):
                txt = h.get_text(strip=True)
                if txt and not any(p in txt.lower() for p in _excluir):
                    empresa = txt
                    break

            items.append({
                "url_detalle"        : url_detalle,
                "cargo_raw"          : cargo.get_text(strip=True) if cargo else "",
                "empresa_raw"        : empresa,
                "ubicacion_raw"      : ubicacion,
                "modalidad_raw"      : modalidad,
                "portal_nombre"      : "multitrabajos",
                "fecha_extraccion"   : datetime.now().isoformat(),
                "vacantes_num"       : None,
                "descripcion_raw"    : None,
                "requisitos_raw"     : None,
                "beneficios_raw"     : None,
                "contrato_raw"       : None,
                "jornada_raw"        : None,
                "salario_min"        : None,
                "salario_max"        : None,
                "salario_a_convenir" : 1,
                "experiencia_anos"   : None,
                "instruccion_raw"    : None,
                "codigo_dpa"         : None,
                "codigo_ciiu"        : None,
                "codigo_ciuo"        : None,
                "ciiu_procesado"     : 0,
                "ciuo_procesado"     : 0,
                "xgb_imputado"       : 0,
            })

        print(f"  🔄 Scroll {scroll_n+1}/{n_scrolls}: "
              f"+{nuevas} nuevas | Acumulado: {len(items)}")

    return [i for i in items if i["cargo_raw"]]


def scrape_listado_completo(max_paginas: int = 3) -> list[dict]:
    """
    Multitrabajos pagina con ?page=N (confirmado en HTML).
    20 vacantes por página → max_paginas=15 ≈ 300 vacantes.
    La deduplicación por url_detalle previene duplicados entre ejecuciones.
    """
    driver = crear_driver(headless=True)
    todas  = []
    urls_vistas = set()

    try:
        for pag in range(1, max_paginas + 1):
            url = (URL_LISTADO if pag == 1
                   else f"{URL_LISTADO}?page={pag}")
            print(f"\n🔄 Página {pag}/{max_paginas}: {url}")

            lote = scrape_listado_mt(driver, url, n_scrolls=1)

            # Deduplicar por url_detalle
            nuevas = [i for i in lote if i["url_detalle"] not in urls_vistas]
            if not nuevas:
                print(f"  🏁 Sin vacantes nuevas en página {pag}. Fin.")
                break
            for i in nuevas:
                urls_vistas.add(i["url_detalle"])
            todas.extend(nuevas)
            print(f"  ✅ +{len(nuevas)} nuevas | Acumulado: {len(todas)}")
            espera_humana(2, 3)

    finally:
        driver.quit()
        print("\n🏁 Driver cerrado (Paso 1).")

    return todas

print("✅ Celda 6 definida.")

✅ Celda 6 definida.


In [ ]:
# ============================================================
# CELDA 7: PASO 2 — Visitar URL de detalle y extraer
#          campos NO disponibles en el listado
# ============================================================
import re

def _recolectar_siblings(tag) -> list[str]:
    """Recolecta texto de los hermanos siguientes hasta otro título."""
    _secciones = ["descripción", "descripcion", "requisitos",
                  "beneficios", "condiciones", "ofrecemos", "funciones",
                  "presencial", "full-time", "part-time", "indeterminado",
                  "vacante disponible", "publicado", "postulación"]
    textos = []
    for sibling in tag.find_next_siblings():
        texto = sibling.get_text(separator="\n", strip=True)
        if any(s in texto.lower() for s in _secciones):
            break
        if texto:
            textos.append(texto)
        if len(textos) > 10:
            break
    return textos

def extraer_seccion_por_titulo(soup: BeautifulSoup, titulo: str) -> str:
    """
    Busca el tag título y extrae el contenido siguiente.
    Sube al padre si el título está anidado en un contenedor.
    """
    tag_titulo = soup.find(
        lambda tag: tag.name in ["h2","h3","h4","p","span","div","b","strong"]
        and titulo.lower() in tag.get_text(strip=True).lower()
        and len(tag.get_text(strip=True)) < 60
    )
    if not tag_titulo:
        return ""

    # Intentar siblings del tag directo
    textos = _recolectar_siblings(tag_titulo)

    # Si vacío → subir un nivel (el tag está dentro de un contenedor div)
    if not textos and tag_titulo.parent:
        textos = _recolectar_siblings(tag_titulo.parent)

    return "\n".join(textos)




def extraer_salario(texto: str) -> dict:
    """
    Extrae salario desde texto libre.
    Cubre: "$1.200", "$800-$1.500", "USD 1000", "Sueldo base $1.200"
    """
    if not texto:
        return {"salario_min": None, "salario_max": None, "salario_a_convenir": 1}

    t = texto.replace(".", "").replace(",", ".")

    # Rango
    m = re.search(
        r'\$\s*(\d{2,6}(?:\.\d{1,2})?)\s*[-–]\s*\$?\s*(\d{2,6}(?:\.\d{1,2})?)', t
    )
    if m:
        return {"salario_min": float(m.group(1)),
                "salario_max": float(m.group(2)),
                "salario_a_convenir": 0}
    # Único
    m = re.search(r'(?:\$|USD\s*)(\d{2,6}(?:\.\d{1,2})?)', t)
    if m:
        v = float(m.group(1))
        return {"salario_min": v, "salario_max": v, "salario_a_convenir": 0}

    return {"salario_min": None, "salario_max": None,
            "salario_a_convenir": int("convenir" in texto.lower()
                                      or "negoci" in texto.lower())}


def extraer_experiencia(texto: str) -> int | None:
    if not texto:
        return None
    patrones = [
        r'(\d+)\s*\+?\s*a[ñn]os?\s+(?:mínimo|de experiencia|comprobable)',
        r'(?:mínimo|m[ií]nimo)\s+(\d+)\s*a[ñn]os?',
        r'experiencia\s+(?:de\s+)?(\d+)\s*a[ñn]os?',
        r'(\d+)\s*a[ñn]os?\s+(?:de\s+)?experiencia',
    ]
    for p in patrones:
        m = re.search(p, texto.lower())
        if m:
            return int(m.group(1))
    return None


def extraer_instruccion(texto: str) -> str:
    if not texto:
        return ""
    t = texto.lower()
    if any(p in t for p in ["bachiller", "bachillerato", "secundari"]):
        return "Bachiller"
    if any(p in t for p in ["tecnólog", "técnico superior", "tecnico superior"]):
        return "Tecnólogo"
    if any(p in t for p in ["título superior", "tercer nivel", "ingeniería",
                              "ingenieria", "licenciatura", "universitari"]):
        return "Tercer nivel"
    if any(p in t for p in ["maestría", "master", "posgrado", "mba"]):
        return "Cuarto nivel"
    return ""


def parse_jornada_contrato(texto: str):
    """
    'Full-time, Indeterminado' → ('Full-time', 'Indeterminado')
    'Part-time, Ocasional'     → ('Part-time',  'Ocasional')
    """
    if not texto:
        return "", ""
    partes = [p.strip() for p in texto.split(",")]
    jornada  = partes[0] if len(partes) > 0 else texto
    contrato = partes[1] if len(partes) > 1 else ""
    return jornada, contrato


def scrape_detalle_mt(driver, url: str) -> dict:
    """
    Visita la URL de detalle de UNA vacante y extrae
    los campos no disponibles en el listado.

    Usa navegación por:
    1. Atributo name= de íconos (estable)
    2. Texto visible de títulos de sección (estable)
    3. Regex sobre texto completo (fallback robusto)
    """
    driver.get(url)
    try:
        WebDriverWait(driver, 25).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "h1, h2"))
        )
    except TimeoutException:
        print(f"    ⚠️ Timeout en detalle: {url}")
        time.sleep(10)

    espera_humana(2, 4)
    soup = BeautifulSoup(driver.page_source, "html.parser")

    # ── Función auxiliar: ícono → texto adyacente ────────────
    # Estrategia: el <p> siempre es hermano inmediato del <i>
    # Funciona con y sin <a> wrapper → más robusto que buscar li padre
    def icono_texto(name: str) -> str:
        i_tag = soup.find("i", attrs={"name": name})
        if not i_tag:
            return ""
        # Caso 1: <i><p> son hermanos directos (clock, people)
        p = i_tag.find_next_sibling("p")
        if p:
            return p.get_text(strip=True)
        # Caso 2: <a><i><p></a> — p está dentro del mismo padre que i
        parent = i_tag.parent
        if parent:
            p = parent.find("p")
            if p:
                return p.get_text(strip=True)
        return ""

    # Buscar patrón "N vacante(s) disponible(s)" directamente en el DOM
    # Más robusto que depender del ícono específico
    tag_vac = soup.find(
        lambda t: t.name in ["p", "span", "li", "h2", "h3"]
        and "vacante" in t.get_text(strip=True).lower()
        and len(t.get_text(strip=True)) < 50
    )
    vacantes_texto = tag_vac.get_text(strip=True) if tag_vac else ""

    if "múltiple" in vacantes_texto.lower() or "multiple" in vacantes_texto.lower():
        vacantes_num = 99   # flag: múltiples sin número exacto
    else:
        m_vac        = re.search(r'(\d+)', vacantes_texto)
        vacantes_num = int(m_vac.group(1)) if m_vac else 1

    # ── Metadatos por ícono ───────────────────────────────────
    #Se sugiere revisar el codigo de la pagina e inspeccionar el icono
    # Contrato: puede venir de ícono propio O embebido en jornada
    _jornada_crudo  = icono_texto("icon-light-clock")
    _contrato_icono = icono_texto("icon-light-document-signed")
    jornada, contrato_parseado = parse_jornada_contrato(_jornada_crudo)
    # Preferir el ícono explícito; si vacío, usar el parseado del clock
    contrato = _contrato_icono if _contrato_icono else contrato_parseado
    area       = icono_texto("icon-light-briefcase")
    idioma     = icono_texto("icon-light-language")
    # Subarea: ícono icon-light-cube → <p> hermana dentro del <a>
    subarea = ""
    i_cube = soup.find("i", attrs={"name": "icon-light-cube"})
    if i_cube:
        a_padre = i_cube.find_parent("a")
        if a_padre:
            p_texto = a_padre.find("p")
            if p_texto:
                subarea = p_texto.get_text(strip=True)

    # Industria: <p> o <span> que diga "Industria" → tomar el siguiente hermano
    industria = ""
    tag_ind = soup.find(
        lambda tag: tag.name in ["p", "span", "div", "b", "strong", "h4"]
        and tag.get_text(strip=True).lower() == "industria"
    )
    if tag_ind:
        siguiente = tag_ind.find_next_sibling(["p", "span", "div"])
        if siguiente:
            industria = siguiente.get_text(strip=True)
        # Fallback: el texto del padre inmediato
        elif tag_ind.parent:
            textos_padre = [
                t.get_text(strip=True)
                for t in tag_ind.parent.find_all(["p","span"])
                if t.get_text(strip=True).lower() != "industria"
            ]
            industria = textos_padre[0] if textos_padre else ""
    licencia   = icono_texto("icon-light-car")

    # ── Secciones por título visible ─────────────────────────
    descripcion = extraer_seccion_por_titulo(soup, "Descripción")
    requisitos  = extraer_seccion_por_titulo(soup, "Requisitos")
    beneficios  = extraer_seccion_por_titulo(soup, "Beneficios")

    # Texto completo para extracción de salario/experiencia/instrucción
    # Si las secciones están vacías, usar texto_raw como base de extracción
    
    # ── Texto completo raw (para minería posterior) ───────────
    # Captura el contenedor principal del detalle de la vacante
    # Fallback en cascada: sección específica → body completo
    contenedor = (
        soup.find("div", id="section-detalle")
        or soup.find("div", id="ficha-detalle")
        or soup.find("p", attrs={"class": lambda c: c and "fSZaM" in " ".join(c)})
        or soup.find("main")
        or soup.body
    )
    texto_raw = contenedor.get_text(separator="\n", strip=True) if contenedor else ""
    texto_completo = (f"{descripcion} {requisitos} {beneficios}".strip()
                      or texto_raw)
    sal          = extraer_salario(texto_completo)
    exp_anos     = extraer_experiencia(texto_completo)
    instruccion  = extraer_instruccion(texto_completo)

    return {
        "vacantes_num"      : vacantes_num,
        "descripcion_raw"   : descripcion,
        "requisitos_raw"    : requisitos,
        "beneficios_raw"    : beneficios,
        "contrato_raw"      : contrato,
        "jornada_raw"       : jornada,
        "area_raw"          : area,
        "idioma_raw"        : idioma,
        "subarea_raw"       : subarea,
        "texto_raw"         : texto_raw,
        "industria_raw"     : industria,
        "licencia_raw"      : licencia,
        "salario_min"       : sal["salario_min"],
        "salario_max"       : sal["salario_max"],
        "salario_a_convenir": sal["salario_a_convenir"],
        "experiencia_anos"  : exp_anos,
        "instruccion_raw"   : instruccion,
    }


print("✅ Celda 7 definida.")

✅ Celda 7 definida.


In [65]:
# ============================================================
# CELDA 8: Pipeline integrado
# ============================================================

def pipeline_multitrabajos(max_paginas: int = 2):
    """
    Paso 1 → listado → URLs + campos básicos
    Paso 2 → detalle → campos completos
    Resultado → lista consolidada lista para la BD
    """
    # ── PASO 1 ────────────────────────────────────────────────
    print("="*55)
    print("PASO 1: Scraping del listado")
    print("="*55)
    items = scrape_listado_completo(max_paginas=max_paginas)
    print(f"\n📋 URLs obtenidas: {len(items)}")

    # ── PASO 2 ────────────────────────────────────────────────
    print("\n" + "="*55)
    print("PASO 2: Scraping de detalles")
    print("="*55)

    driver   = crear_driver()
    errores  = 0

    try:
        for idx, item in enumerate(items, 1):
            url = item.get("url_detalle", "")
            if not url:
                continue

            print(f"  [{idx}/{len(items)}] {item['cargo_raw'][:45]}...")
            try:
                detalle = scrape_detalle_mt(driver, url)
                item.update(detalle)   # merge Paso1 + Paso2
            except Exception as e:
                errores += 1
                print(f"    ⚠️  Error: {e}")

            espera_humana(2, 4)

    finally:
        driver.quit()
        print("\n🏁 Driver cerrado (Paso 2).")

    # ── REPORTE ───────────────────────────────────────────────
    import pandas as pd
    df = pd.DataFrame(items)

    print(f"\n{'='*55}")
    print(f"RESULTADO: {len(df)} vacantes | {errores} errores")
    print(f"{'='*55}")

    campos = ["cargo_raw","empresa_raw","ubicacion_raw","modalidad_raw",
              "vacantes_num","contrato_raw","jornada_raw",
              "salario_min","experiencia_anos","instruccion_raw"]

    print("\n📊 Cobertura real de campos (excluye vacíos):")
    for campo in campos:
        if campo in df.columns:
            llenos = df[campo].apply(
                lambda x: x is not None
                          and pd.notna(x)
                          and str(x).strip() not in ("", "None", "nan", "0")
            ).sum()
            cob = llenos / len(df) * 100
            bar = "█" * int(cob/5) + "░" * (20 - int(cob/5))
            print(f"  {campo:<22} {bar} {cob:5.1f}%  ({llenos}/{len(df)})")


    if not df.empty:
        print(f"\n🔎 Primera vacante completa:")
        for k, v in items[0].items():
            print(f"  {k:<22}: {str(v)[:75]}")

    return items

# ── EJECUTAR ──────────────────────────────────────────────────
# n_scrolls=3 → ~60 vacantes únicas (prueba rápida ~3 min)
# n_scrolls=20 → ~400 vacantes (recolección masiva)
vacantes_mt = pipeline_multitrabajos(max_paginas=3)

PASO 1: Scraping del listado

🔄 Iniciando scroll infinito (3 scrolls)...
  🔄 Scroll 1/3: +20 nuevas | Acumulado: 20
  🔄 Scroll 2/3: +0 nuevas | Acumulado: 20
  🔄 Scroll 3/3: +0 nuevas | Acumulado: 20
  🏁 2 scrolls sin nuevas vacantes. Fin de página.
  ✅ Total único: 20

🏁 Driver cerrado (Paso 1).

📋 URLs obtenidas: 20

PASO 2: Scraping de detalles
  [1/20] ANALISTA DE CONTROL DE CALIDAD...
  [2/20] Cajero/a...
  [3/20] Pasante de Talento Humano y selección-medio t...
  [4/20] ASISTENTE DE CALL CENTER - BILINGUE...
  [5/20] Analista de Compras...
  [6/20] JEFE DE CONTROL INTERNO Y COMPLIANCE...
  [7/20] Contador General con experiencia en Costos...
  [8/20] Analista de ambiente...
  [9/20] AUXILIAR DE TRAMITES E INVENTARIOS...
  [10/20] ASESOR COMERCIAL...
  [11/20] Asistente Contable...
  [12/20] Auxiliar de Bodega...
  [13/20] COCINERO / A...
  [14/20] ANALISTA CONTABLE/FINANCIERO...
  [15/20] JEFE COMERCIAL B2B - QUITO...
  [16/20] JEFE DE MANTENIMIENTO...
  [17/20] Asistente de Comp